# 06 - Point Location

        Source span: printed pages 121-146; PDF pages 132-157. This notebook is original standalone study material for *Computational Geometry: Algorithms and Applications*, Third Edition. It uses the book as orientation for chapter structure and concepts, but it does not copy textbook prose, exercises, screenshots, page crops, or figures.

        ## Standalone Goal

        The chapter question is: how do we turn finding which subdivision face contains a query point into a precise geometric algorithm that can be inspected, tested, and implemented? The answer is not just a theorem statement. It is a chain of modeling decisions: choose the geometric primitive, identify the invariant that makes the primitive useful, pick a data structure that exposes only the necessary local information, and verify that the resulting algorithm reports the same answer as a slower direct method on small examples.

        This notebook teaches trapezoidal maps, randomized incremental construction, search DAG, degeneracies. The diagrams are not decoration; each one is a small laboratory. The first visual fixes the geometric object of study. The second visual exposes the algorithmic state that changes over time. The computational check at the end records the invariant that should survive those state changes. When an algorithm is randomized, the notebook uses a deterministic seed so the displayed run is reproducible while still showing what changes under a random order. When an algorithm is sensitive to degeneracy, the notebook names the fragile predicate instead of hiding it inside a library call.

        A recurring theme in computational geometry is that the obvious mathematical definition is often too large to compute directly. A convex hull is an intersection of all convex sets, but the algorithm works with turns in a sorted list of points. A subdivision may be a planar set, but an overlay algorithm needs vertex, edge, face, and incidence records. A nearest-site region is defined by infinitely many distances, but a diagram is built from bisectors and events. For this chapter, the same translation happens through randomized incremental trapezoidal map with a directed acyclic search structure. The notebook therefore keeps two views in sync: the continuous geometric object and the finite record that an algorithm can update.

        ## Translation Guide

        | Textbook idea | Computational translation |
        | --- | --- |
        | Application frame | finding which subdivision face contains a query point |
| Geometric problem | trapezoidal maps, randomized incremental construction, search DAG, degeneracies |
| Algorithmic core | randomized incremental trapezoidal map with a directed acyclic search structure |
| Data structures | trapezoids, neighbor pointers, search DAG leaves |
| Visual model | point location |

        ## Route Through The Chapter

        1. decompose segments into vertical trapezoids.
2. walk a query point through x-nodes and segment tests.
3. show how one inserted segment replaces crossed trapezoids.
4. measure expected search depth over random insertion orders.

        ## Visual Storyboard

        The visual sequence follows a fixed teaching rhythm. First, a concept diagram labels the input geometry and the claimed output. Second, an algorithm-state diagram shows the local decision that lets the algorithm avoid brute force. Third, a small metric table or JSON check records the invariant in numbers. Some chapters also add an interactive HTML artifact when rotation, ordering, or query movement is easier to inspect dynamically than in a single static figure.

        The source chapter's application is treated as motivation rather than as a turnkey software product. The notebook abstracts the application into a compact test instance, because a clean instance makes the correctness argument visible. For example, a map-overlay chapter should display the sweep status and then test it against brute-force intersections; a motion-planning chapter should display configuration obstacles and then test the route against collision predicates; a range-searching chapter should show the query rectangle and then compare the data-structure report with direct filtering.

        ## Worked Examples And Pitfalls

        The worked examples deliberately use small data. Small examples make it possible to see every event, cell, edge, or predicate outcome without trusting a black box. That is also how the notebook handles robustness. A geometric algorithm usually depends on predicates such as orientation, in-circle tests, distance comparison, visibility, or containment. If the predicate is unstable near degeneracy, the notebook displays a near-degenerate case and records a margin. The margin is not a proof of robust industrial arithmetic, but it tells the reader where exact arithmetic or symbolic perturbation would become relevant.

        The starred material for this chapter is treated as an advanced lens: tail estimate. The notebook includes the starred idea as an optional computational extension whenever it changes the geometric picture. If it is mainly analytical, the extension becomes a small experiment: vary input size, random order, or query shape, then compare the measured state count with the stated asymptotic behavior. The point is to let the theorem leave a trace in an artifact, not to replace the proof with a picture.

        ## Implementation Lens

        The implementation is intentionally modest and inspectable. It favors plain arrays, short helper functions, and explicit predicates over a hidden production geometry kernel. That makes the notebook useful for learning: when a result changes, you can usually point to the exact comparison or update that changed it. The price is that these examples are teaching implementations, not industrial robust-geometry packages. The notebook therefore separates two claims. First, the displayed construction is checked on its own sample data. Second, the chapter explains what a complete implementation would still need for hostile inputs: exact predicates, careful event tie-breaking, balanced trees with stable keys, topology records, or certified numerical solvers.

        Read the final JSON artifact as a compact contract. It records the number of objects constructed, the key invariant, and the agreement with a direct check when a direct check is affordable. If you extend the notebook, keep that contract alive. Add a harder instance, then add a check that would fail if the geometric idea were misunderstood. For Point Location, a good extension keeps the same application frame but changes the input enough to stress trapezoids, neighbor pointers, search DAG leaves. The goal is not to produce a large library in one notebook; the goal is to make the algorithm's finite state visible and falsifiable.

        ## Applied Lab

        The applied lab asks you to modify the sample instance while keeping the final checks true. Change a site, obstacle, segment, query window, or constraint. Then re-run the notebook and inspect which artifact changes first. If the answer changes but the invariant still holds, the model is doing useful work. If the invariant fails, the failure usually points to exactly the geometric assumption that the algorithm needs: general position, non-crossing input, convexity, sorted order, balanced refinement, or a complete visibility test.

        ## Sanity Checks To Read

        - each query lands in exactly one toy trapezoid.
- search path decisions agree with direct containment tests.
- randomized depth summary stays below the linear scan baseline.

        ## Takeaways

        By the end of the notebook, you should be able to explain the chapter without opening the PDF: what geometric object is being computed, which finite state represents it, why the state changes are local, which degeneracies matter, and what numerical or combinatorial check confirms the displayed result. That is the standard for this course: a chapter is complete when its prose, code, visuals, and checks together carry the computational geometry.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Computational-Geometry-Algorithms-and-Applications/chapter-06-point-location/06-point-location.ipynb",
  "course_dir": "Computational-Geometry-Algorithms-and-Applications",
  "course_title": "Computational Geometry Algorithms and Applications",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Computational-Geometry-Algorithms-and-Applications/chapter-06-point-location/06-point-location.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Computational-Geometry-Algorithms-and-Applications/chapter-06-point-location/06-point-location.ipynb",
  "notebook_title": "06 - Point Location",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/graphics.txt",
  "runtime_profile": "graphics"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the CGAA book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, save_json
from utils.chapter_visuals import build_chapter_visuals, chapter_lab_summary

ARTIFACT_ROOT = BOOK_ROOT / "artifacts"


In [ ]:
source_span = json.loads('{\n  "number": 6,\n  "label": "Chapter 06",\n  "title": "Point Location",\n  "subtitle": "Knowing Where You Are",\n  "folder": "chapter-06-point-location",\n  "notebook": "06-point-location.ipynb",\n  "printed_pages": "121-146",\n  "pdf_pages": "132-157",\n  "focus": "trapezoidal maps, randomized incremental construction, search DAG, degeneracies",\n  "application": "finding which subdivision face contains a query point",\n  "algorithmic_core": "randomized incremental trapezoidal map with a directed acyclic search structure",\n  "data_structures": "trapezoids, neighbor pointers, search DAG leaves",\n  "starred": "tail estimate",\n  "visual_kind": "point-location",\n  "route": [\n    "decompose segments into vertical trapezoids",\n    "walk a query point through x-nodes and segment tests",\n    "show how one inserted segment replaces crossed trapezoids",\n    "measure expected search depth over random insertion orders"\n  ],\n  "checks": [\n    "each query lands in exactly one toy trapezoid",\n    "search path decisions agree with direct containment tests",\n    "randomized depth summary stays below the linear scan baseline"\n  ],\n  "artifact_topic": "chapter-06"\n}')
CHAPTER_TOPIC = source_span["artifact_topic"]
CHAPTER_ARTIFACT_ROOT = ARTIFACT_ROOT / CHAPTER_TOPIC
CHAPTER_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
source_span


In [ ]:
visual_results = build_chapter_visuals(source_span, ARTIFACT_ROOT)
for item in visual_results["artifacts"]:
    display_artifact(BOOK_ROOT / item["relative_path"], width=760)
visual_results["summary"]


In [ ]:
lab_summary = chapter_lab_summary(source_span)
lab_summary


In [ ]:
final_sanity = {
    "chapter": source_span["label"],
    "title": source_span["title"],
    "source_span": {
        "printed_pages": source_span["printed_pages"],
        "pdf_pages": source_span["pdf_pages"],
    },
    "artifact_count": len(visual_results["artifacts"]),
    "visual_summary": visual_results["summary"],
    "lab_summary": lab_summary,
    "checks": source_span["checks"],
}
check_path = save_json(final_sanity, CHAPTER_ARTIFACT_ROOT / "checks" / "final-sanity.json")
required_paths = [BOOK_ROOT / item["relative_path"] for item in visual_results["artifacts"]]
required_paths.append(check_path)
assert_artifacts(required_paths)
final_sanity
